# Student Performance Big Data Analytics Pipeline
## Improving Student Performance Using Big Data Analytics and Intelligent Recommendation Systems

**Reference Paper:**
> Manoharan et al. (2025). *Big data and analytics in education: Leveraging data to improve student performance.* IEEE WorldSUAS. DOI: 10.1109/WorldSUAS66815.2025.1119913

---


## 0. Imports & Setup

In [4]:
import os, time, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from collections import defaultdict

# Install vaderSentiment if not already installed
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score, accuracy_score)
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
np.random.seed(42)

GRAPHS_DIR = "graphs"
os.makedirs(GRAPHS_DIR, exist_ok=True)
RESULTS = {}

# Colour palette
NAVY   = "#1A3A5C"
TEAL   = "#0D9488"
TEAL_L = "#14B8A6"
RED    = "#EF4444"
AMBER  = "#F59E0B"
GREEN  = "#22C55E"
GRAY   = "#64748B"
LGRAY  = "#E2E8F0"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})
print("Setup complete.")

Setup complete.


## 1. Dataset Generation

A synthetic dataset of **1,200 student records** is generated using realistic statistical distributions,
satisfying the project requirement of ≥1,000 data items.

**Features:** attendance percentage, midterm score, quiz average, assignment score, forum posts, final score, label (At-Risk / Satisfactory / Excellence), and free-text feedback.

In [5]:
N = 1200
subjects = ["Mathematics", "Physics", "Chemistry", "Computer Science", "Statistics"]
courses  = [f"CS10{i}" for i in range(1, 9)] + [f"MA20{i}" for i in range(1, 5)]

attendance = np.clip(np.random.normal(75, 15, N), 20, 100)
midterm    = np.clip(np.random.normal(68, 18, N),  0, 100)
quiz_avg   = np.clip(midterm + np.random.normal(0, 10, N), 0, 100)
assignment = np.clip(np.random.normal(80, 12, N),  0, 100)
forum_post = np.random.poisson(5, N)

final_score = np.clip(
    0.30*midterm + 0.25*attendance + 0.20*quiz_avg + 0.15*assignment
    + 0.10*(forum_post/forum_post.max()*100) + np.random.normal(0, 5, N),
    0, 100
)

def label_student(score, att):
    if score < 55 or att < 50:       return "At-Risk"
    if score >= 80 and att >= 80:    return "Excellence"
    return "Satisfactory"

labels = [label_student(final_score[i], attendance[i]) for i in range(N)]

pos_phrases = [
    "The course was very engaging and well structured.",
    "I really enjoyed the assignments, very helpful.",
    "Excellent instructor, clear explanations every class.",
    "Loved the interactive sessions and group projects.",
    "Great material, learned a lot from this course.",
    "The professor is very approachable and knowledgeable.",
    "Assignments were challenging but rewarding.",
    "One of the best courses I have taken so far.",
]
neu_phrases = [
    "The course content was average, could be improved.",
    "Lectures were okay but reading material was unclear.",
    "Some topics were interesting, others not so much.",
    "The course met expectations, nothing special.",
    "Assignments were manageable but lacked depth.",
    "Standard course, nothing extraordinary to report.",
]
neg_phrases = [
    "The course was very confusing and poorly organised.",
    "I struggled to understand the topics, no support.",
    "Too much workload with very little guidance given.",
    "Instructor was often unavailable and unhelpful.",
    "The content was outdated and not relevant at all.",
    "I felt lost most of the time during the lectures.",
    "Very disappointing course, needs major improvement.",
]
all_phrases = pos_phrases + neu_phrases + neg_phrases

feedback_raw = [
    all_phrases[np.random.choice(
        len(pos_phrases) if labels[i] == "Excellence"
        else (range(len(pos_phrases), len(pos_phrases)+len(neu_phrases))
              if labels[i] == "Satisfactory"
              else range(len(pos_phrases)+len(neu_phrases), len(all_phrases)))
    )] for i in range(N)
]

df = pd.DataFrame({
    "student_id":       [f"STU{1000+i}" for i in range(N)],
    "subject":          np.random.choice(subjects, N),
    "course_id":        np.random.choice(courses, N),
    "attendance_pct":   np.round(attendance, 1),
    "midterm_score":    np.round(midterm, 1),
    "quiz_avg":         np.round(quiz_avg, 1),
    "assignment_score": np.round(assignment, 1),
    "forum_posts":      forum_post,
    "final_score":      np.round(final_score, 1),
    "label":            labels,
    "feedback":         feedback_raw,
})

df.to_csv("student_dataset.csv", index=False)
print(f"Dataset shape : {df.shape}")
print(f"Label dist    : {df['label'].value_counts().to_dict()}")
RESULTS["n_records"]  = N
RESULTS["label_dist"] = df["label"].value_counts().to_dict()
df.head()

Dataset shape : (1200, 11)
Label dist    : {'Satisfactory': 929, 'At-Risk': 171, 'Excellence': 100}


,student_id,subject,course_id,attendance_pct,midterm_score,quiz_avg,assignment_score,forum_posts,final_score,label,feedback
0,STU1000,Computer Science,MA201,82.5,70.3,63.0,70.1,5,77.4,Satisfactory,Assignments were manageable but lacked depth.
1,STU1001,Computer Science,CS103,72.9,60.3,62.0,71.4,3,64.5,Satisfactory,"The course met expectations, nothing special."
2,STU1002,Chemistry,CS101,84.7,70.2,64.7,100.0,6,62.0,Satisfactory,Lectures were okay but reading material was un...
3,STU1003,Statistics,MA203,97.8,77.8,75.1,80.0,7,85.9,Excellence,The professor is very approachable and knowled...
4,STU1004,Mathematics,MA203,71.5,68.9,85.6,89.4,3,76.7,Satisfactory,Lectures were okay but reading material was un...


## 2. Task 1 — MapReduce Simulation (Big Data Processing)

A MapReduce-style pipeline processes the student dataset using a **Mapper** that emits `(subject, score)` key-value pairs
and a **Reducer** that computes average performance per subject and counts at-risk students.

We also benchmark **MapReduce vs. traditional Pandas** processing across increasing dataset sizes.

In [6]:
def mapreduce_avg_by_subject(dataframe):
    """Mapper: emit (subject, score). Reducer: mean per subject."""
    mapped = defaultdict(list)
    for _, row in dataframe.iterrows():
        mapped[row["subject"]].append(row["final_score"])
    return {subject: np.mean(scores) for subject, scores in mapped.items()}

def mapreduce_at_risk_count(dataframe):
    """Mapper: emit (subject, is_at_risk). Reducer: count at-risk per subject."""
    mapped = defaultdict(int)
    for _, row in dataframe.iterrows():
        if row["label"] == "At-Risk":
            mapped[row["subject"]] += 1
    return dict(mapped)

# Benchmark MapReduce vs. Pandas
sizes = [100, 300, 600, 1000, 1200]
mr_times, pd_times = [], []

for sz in sizes:
    subset = df.head(sz)
    t0 = time.perf_counter()
    _ = mapreduce_avg_by_subject(subset)
    mr_times.append((time.perf_counter() - t0) * 1000)

    t0 = time.perf_counter()
    _ = subset.groupby("subject")["final_score"].mean()
    pd_times.append((time.perf_counter() - t0) * 1000)

mr_results = mapreduce_avg_by_subject(df)
atrisk_cnt = mapreduce_at_risk_count(df)

RESULTS["mr_subject_avg"]    = {k: round(v, 2) for k, v in mr_results.items()}
RESULTS["atrisk_by_subject"] = atrisk_cnt

print("Average Final Score by Subject (MapReduce Reducer output):")
for subj, avg in sorted(mr_results.items()):
    print(f"  {subj:25s}: {avg:.2f}  |  At-Risk: {atrisk_cnt.get(subj,0)}")

Average Final Score by Subject (MapReduce Reducer output):
  Chemistry                : 67.98  |  At-Risk: 33
  Computer Science         : 68.48  |  At-Risk: 37
  Mathematics              : 68.01  |  At-Risk: 28
  Physics                  : 67.29  |  At-Risk: 40
  Statistics               : 68.68  |  At-Risk: 33


In [7]:
# Figure 1 – MapReduce scalability & subject averages
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(sizes, mr_times, marker="o", color=TEAL,  lw=2.5, label="MapReduce (Python)")
axes[0].plot(sizes, pd_times, marker="s", color=NAVY,  lw=2.5, label="Traditional (Pandas)")
axes[0].fill_between(sizes, mr_times, pd_times, alpha=0.08, color=TEAL)
axes[0].set_xlabel("Dataset Size (records)", fontsize=11)
axes[0].set_ylabel("Processing Time (ms)",   fontsize=11)
axes[0].set_title("Scalability: MapReduce vs. Traditional", fontsize=13, fontweight="bold", color=NAVY)
axes[0].legend(fontsize=10)

subjects_sorted = sorted(mr_results, key=mr_results.get)
avgs   = [mr_results[s] for s in subjects_sorted]
colors = [TEAL if v >= np.mean(avgs) else NAVY for v in avgs]
bars   = axes[1].barh(subjects_sorted, avgs, color=colors, edgecolor="white", height=0.6)
for bar, val in zip(bars, avgs):
    axes[1].text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                 f"{val:.1f}", va="center", fontsize=10, color=GRAY)
axes[1].set_xlabel("Average Final Score", fontsize=11)
axes[1].set_title("Avg Score by Subject (Reducer Output)", fontsize=13, fontweight="bold", color=NAVY)
axes[1].set_xlim(0, 105)

plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/fig1_mapreduce.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 1 saved.")

Figure 1 saved.


## 3. Task 2 — Predictive Modeling & Early Warning System

Three classifiers are trained to categorise students into **At-Risk**, **Satisfactory**, or **Excellence**:
- Random Forest
- Gradient Boosting
- Logistic Regression

**Features:** attendance %, midterm score, quiz average, assignment score, forum post count.
**Evaluation:** 75/25 stratified split, confusion matrix, precision, recall, F1, accuracy.

In [8]:
features = ["attendance_pct", "midterm_score", "quiz_avg", "assignment_score", "forum_posts"]
X  = df[features].values
le = LabelEncoder()
y  = le.fit_transform(df["label"])   # At-Risk=0, Excellence=1, Satisfactory=2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

models = {
    "Random Forest":       RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
}

model_metrics = {}
for name, clf in models.items():
    X_tr = X_train if "Forest" in name or "Boost" in name else X_train_sc
    X_te = X_test  if "Forest" in name or "Boost" in name else X_test_sc
    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)
    model_metrics[name] = {
        "precision": round(precision_score(y_test, y_pred, average="weighted"), 4),
        "recall":    round(recall_score   (y_test, y_pred, average="weighted"), 4),
        "f1":        round(f1_score       (y_test, y_pred, average="weighted"), 4),
        "accuracy":  round(accuracy_score (y_test, y_pred), 4),
    }
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

RESULTS["model_metrics"] = model_metrics


Random Forest
              precision    recall  f1-score   support

     At-Risk       0.80      0.56      0.66        43
  Excellence       0.75      0.60      0.67        25
Satisfactory       0.88      0.95      0.92       232

    accuracy                           0.87       300
   macro avg       0.81      0.70      0.75       300
weighted avg       0.86      0.87      0.86       300


Gradient Boosting
              precision    recall  f1-score   support

     At-Risk       0.79      0.63      0.70        43
  Excellence       0.76      0.64      0.70        25
Satisfactory       0.90      0.95      0.92       232

    accuracy                           0.88       300
   macro avg       0.82      0.74      0.77       300
weighted avg       0.87      0.88      0.87       300


Logistic Regression
              precision    recall  f1-score   support

     At-Risk       0.68      0.58      0.62        43
  Excellence       0.76      0.64      0.70        25
Satisfactory       0

In [9]:
# Figure 2 – Model performance comparison, confusion matrix, feature importance
rf_model  = models["Random Forest"]
feat_imp  = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
y_pred_rf = rf_model.predict(X_test)
cm        = confusion_matrix(y_test, y_pred_rf)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metric_names = ["Precision", "Recall", "F1-Score", "Accuracy"]
x = np.arange(len(metric_names))
w = 0.25
for i, (mname, mvals) in enumerate(model_metrics.items()):
    vals = [mvals["precision"], mvals["recall"], mvals["f1"], mvals["accuracy"]]
    axes[0].bar(x + i*w, vals, w, label=mname,
                color=[TEAL, NAVY, GRAY][i], alpha=0.9)
axes[0].set_xticks(x + w); axes[0].set_xticklabels(metric_names, fontsize=10)
axes[0].set_ylim(0.5, 1.05); axes[0].set_ylabel("Score", fontsize=11)
axes[0].set_title("Model Performance Comparison", fontsize=12, fontweight="bold", color=NAVY)
axes[0].legend(fontsize=8)
axes[0].yaxis.grid(True, alpha=0.3); axes[0].set_axisbelow(True)
for bg in axes[0].containers:
    axes[0].bar_label(bg, fmt="%.2f", fontsize=7, padding=1)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=axes[1], linewidths=0.5, linecolor="white", annot_kws={"size":12})
axes[1].set_title("Confusion Matrix (Random Forest)", fontsize=12, fontweight="bold", color=NAVY)
axes[1].set_xlabel("Predicted", fontsize=11); axes[1].set_ylabel("Actual", fontsize=11)

clrs_fi = [TEAL if v == feat_imp.max() else NAVY for v in feat_imp.values]
feat_imp.plot(kind="barh", ax=axes[2], color=clrs_fi, edgecolor="white")
axes[2].set_title("Feature Importance (Random Forest)", fontsize=12, fontweight="bold", color=NAVY)
axes[2].set_xlabel("Importance Score", fontsize=11)
for i, v in enumerate(feat_imp.values):
    axes[2].text(v+0.002, i, f"{v:.3f}", va="center", fontsize=9, color=GRAY)

plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/fig2_predictive_model.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 2 saved.")

Figure 2 saved.


In [10]:
# Figure 3 – Classification distribution & scatter plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

label_counts = df["label"].value_counts()
axes[0].pie(label_counts.values, labels=label_counts.index, autopct="%1.1f%%",
            startangle=90, colors=[RED, AMBER, GREEN],
            wedgeprops={"edgecolor":"white","linewidth":2}, textprops={"fontsize":11})
for at in axes[0].texts[1::2]:
    at.set_fontsize(11); at.set_color("white"); at.set_fontweight("bold")
axes[0].set_title("Student Classification Distribution", fontsize=13, fontweight="bold", color=NAVY)

cmap = {"At-Risk": RED, "Satisfactory": AMBER, "Excellence": GREEN}
for lbl, grp in df.groupby("label"):
    axes[1].scatter(grp["attendance_pct"], grp["final_score"],
                    c=cmap[lbl], alpha=0.45, s=18, label=lbl)
axes[1].set_xlabel("Attendance (%)", fontsize=11); axes[1].set_ylabel("Final Score", fontsize=11)
axes[1].set_title("Attendance vs. Final Score by Label", fontsize=12, fontweight="bold", color=NAVY)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/fig3_classification.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 3 saved.")

Figure 3 saved.


## 4. Task 3 — Collaborative Filtering Recommendation Engine

A **user-based collaborative filtering** engine is built on a 200×12 student–course interaction matrix.

**Process:**
1. Build student–course rating matrix (ratings 1–5, 42% sparsity)
2. Compute cosine similarity between student profiles
3. Identify k=10 nearest-neighbour students
4. Recommend top-N unrated courses weighted by neighbour similarity

**Evaluation:** leave-one-out protocol across Top-N ∈ {3, 5, 7, 10}

In [11]:
unique_students = df["student_id"].unique()[:200]
unique_courses  = df["course_id"].unique()

R_matrix = np.zeros((len(unique_students), len(unique_courses)))
for i in range(len(unique_students)):
    n_rated   = np.random.randint(3, len(unique_courses))
    rated_idx = np.random.choice(len(unique_courses), n_rated, replace=False)
    for j in rated_idx:
        R_matrix[i, j] = np.random.choice([3, 4, 5], p=[0.2, 0.4, 0.4])

sparsity = 100 * (R_matrix == 0).sum() / R_matrix.size
print(f"Interaction matrix: {R_matrix.shape[0]} students × {R_matrix.shape[1]} courses")
print(f"Sparsity: {sparsity:.1f}%")

def user_based_cf(R, target_idx, k=10, top_n=3):
    """Return top_n course indices not yet rated by target student."""
    sim = cosine_similarity(R)[target_idx]
    sim[target_idx] = -1
    neighbors = np.argsort(sim)[::-1][:k]
    scores, weight_sum = np.zeros(R.shape[1]), 0
    for nb in neighbors:
        scores += sim[nb] * R[nb]
        weight_sum += abs(sim[nb])
    if weight_sum > 0:
        scores /= weight_sum
    already_rated = np.where(R[target_idx] > 0)[0]
    scores[already_rated] = -1
    return np.argsort(scores)[::-1][:top_n]

def evaluate_cf(R, k=10, top_n=5, n_test=150):
    """Leave-one-out evaluation."""
    hits = 0
    for i in np.random.choice(R.shape[0], n_test, replace=False):
        rated = np.where(R[i] > 0)[0]
        if len(rated) < 2: continue
        target_item = np.random.choice(rated)
        R_temp = R.copy(); R_temp[i, target_item] = 0
        recs = user_based_cf(R_temp, i, k=k, top_n=top_n)
        if target_item in recs: hits += 1
    recall    = hits / n_test
    precision = recall
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
    return precision, recall, f1

top_n_range = [3, 5, 7, 10]
cf_results  = {}
print(f"{'Top-N':>6}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}")
print("-" * 40)
for tn in top_n_range:
    p, r, f = evaluate_cf(R_matrix, k=10, top_n=tn, n_test=150)
    cf_results[f"top_{tn}"] = {"precision": round(p,4), "recall": round(r,4), "f1": round(f,4)}
    print(f"  top-{tn:2d}  {p:10.4f}  {r:8.4f}  {f:8.4f}")

RESULTS["cf_metrics"] = cf_results

recs = user_based_cf(R_matrix, 0, k=10, top_n=3)
print(f"\nSample recommendations for STU1000: {[unique_courses[r] for r in recs]}")

Interaction matrix: 200 students × 12 courses
Sparsity: 40.8%
 Top-N   Precision    Recall        F1
----------------------------------------
  top- 3      0.5733    0.5733    0.5733
  top- 5      0.8000    0.8000    0.8000
  top- 7      0.9133    0.9133    0.9133
  top-10      1.0000    1.0000    1.0000

Sample recommendations for STU1000: ['CS105', 'CS101', 'CS107']


In [12]:
# Figure 4 – CF metrics & interaction matrix heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

top_ns = [int(k.split("_")[1]) for k in cf_results]
precs  = [v["precision"] for v in cf_results.values()]
recs_  = [v["recall"]    for v in cf_results.values()]
f1s    = [v["f1"]        for v in cf_results.values()]

axes[0].plot(top_ns, precs, "o-",  color=TEAL, lw=2.5, label="Precision")
axes[0].plot(top_ns, recs_, "s--", color=NAVY, lw=2.5, label="Recall")
axes[0].plot(top_ns, f1s,   "^:",  color=RED,  lw=2.5, label="F1-Score")
axes[0].set_xlabel("Top-N Recommendations", fontsize=11)
axes[0].set_ylabel("Score",                 fontsize=11)
axes[0].set_title("CF Metrics vs. Top-N",   fontsize=13, fontweight="bold", color=NAVY)
axes[0].legend(fontsize=10); axes[0].set_ylim(0, 1)

sns.heatmap(R_matrix[:15, :12], ax=axes[1], cmap="YlGnBu",
            xticklabels=[f"C{i}" for i in range(12)],
            yticklabels=[f"S{i}" for i in range(15)],
            linewidths=0.4, linecolor="white", cbar_kws={"label":"Rating"})
axes[1].set_title("Student–Course Interaction Matrix (15×12 sample)",
                  fontsize=12, fontweight="bold", color=NAVY)
axes[1].set_xlabel("Course", fontsize=10); axes[1].set_ylabel("Student", fontsize=10)

plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/fig4_collaborative_filtering.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 4 saved.")

Figure 4 saved.


## 5. Task 4 — VADER Sentiment Analysis & Engagement Tracking

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) scores each student feedback string with a compound score in **[-1.0, +1.0]**:
- Compound ≥ 0.05 → **Positive**
- Compound ≤ -0.05 → **Negative**
- Otherwise → **Neutral**

An **Engagement Index** is derived:
> `Engagement Index = 0.6 × normalised_compound + 0.4 × normalised_forum_posts`

In [13]:
analyser = SentimentIntensityAnalyzer()

def classify_sentiment(compound):
    if compound >= 0.05:  return "Positive"
    if compound <= -0.05: return "Negative"
    return "Neutral"

scores_list     = [analyser.polarity_scores(fb) for fb in df["feedback"]]
df["compound"]  = [s["compound"] for s in scores_list]
df["sentiment"] = df["compound"].apply(classify_sentiment)

norm_comp  = (df["compound"]    - df["compound"].min())    / (df["compound"].max()    - df["compound"].min())
norm_forum = (df["forum_posts"] - df["forum_posts"].min()) / (df["forum_posts"].max() - df["forum_posts"].min())
df["engagement_index"] = 0.6*norm_comp + 0.4*norm_forum

sent_counts       = df["sentiment"].value_counts()
sentiment_by_lbl  = df.groupby("label")["compound"].mean()

RESULTS["sentiment_dist"]     = sent_counts.to_dict()
RESULTS["sentiment_by_label"] = {k: round(v, 4) for k, v in sentiment_by_lbl.items()}

print("Sentiment Distribution:")
print(sent_counts.to_string())
print("\nMean VADER compound score by student label:")
print(sentiment_by_lbl.round(4).to_string())

Sentiment Distribution:
sentiment
Neutral     405
Negative    399
Positive    396

Mean VADER compound score by student label:
label
At-Risk        -0.1602
Excellence      0.5868
Satisfactory    0.0547


In [14]:
# Figure 5 – Sentiment distribution, VADER by label, engagement vs score
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

labs = [f"{k}\n({v})" for k, v in sent_counts.items()]
axes[0].pie(sent_counts.values, labels=labs, autopct="%1.1f%%",
            colors=[GREEN, GRAY, RED], startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":2}, textprops={"fontsize":10})
axes[0].set_title("Sentiment Distribution", fontsize=13, fontweight="bold", color=NAVY)

for lbl, col in [("At-Risk", RED), ("Satisfactory", AMBER), ("Excellence", GREEN)]:
    axes[1].hist(df[df["label"]==lbl]["compound"], bins=20,
                 alpha=0.6, color=col, label=lbl, edgecolor="white")
axes[1].axvline( 0.05, color=GREEN, ls="--", lw=1.2, alpha=0.8)
axes[1].axvline(-0.05, color=RED,   ls="--", lw=1.2, alpha=0.8)
axes[1].set_xlabel("VADER Compound Score", fontsize=11)
axes[1].set_ylabel("Frequency",            fontsize=11)
axes[1].set_title("VADER Scores by Student Category", fontsize=12, fontweight="bold", color=NAVY)
axes[1].legend(fontsize=9)

scatter_c = df["sentiment"].map({"Positive": GREEN, "Neutral": AMBER, "Negative": RED})
axes[2].scatter(df["engagement_index"], df["final_score"], c=scatter_c, alpha=0.4, s=18)
axes[2].set_xlabel("Engagement Index", fontsize=11)
axes[2].set_ylabel("Final Score",      fontsize=11)
axes[2].set_title("Engagement Index vs. Final Score", fontsize=12, fontweight="bold", color=NAVY)
patches = [mpatches.Patch(color=c, label=l)
           for l, c in [("Positive",GREEN),("Neutral",AMBER),("Negative",RED)]]
axes[2].legend(handles=patches, fontsize=9)

plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/fig5_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 5 saved.")

Figure 5 saved.


## 6. Integrated Analytics Dashboard

All four modules combined into a single consolidated visualisation.

In [15]:
fig = plt.figure(figsize=(16, 9))
fig.suptitle("Integrated Student Analytics Dashboard\nBig Data Analytics Project",
             fontsize=16, fontweight="bold", color=NAVY, y=0.98)

ax1 = fig.add_subplot(2, 4, 1)
lc2 = df["label"].value_counts()
ax1.bar(lc2.index, lc2.values, color=[RED, AMBER, GREEN], edgecolor="white", width=0.5)
ax1.set_title("Risk Distribution", fontweight="bold", color=NAVY); ax1.set_ylabel("Count")
for i, (k, v) in enumerate(lc2.items()):
    ax1.text(i, v+5, str(v), ha="center", fontsize=10, fontweight="bold")

ax2 = fig.add_subplot(2, 4, 2)
rf_m = model_metrics["Random Forest"]
bars = ax2.bar(["Precision","Recall","F1","Accuracy"],
               [rf_m["precision"],rf_m["recall"],rf_m["f1"],rf_m["accuracy"]],
               color=[TEAL,TEAL,TEAL,NAVY], edgecolor="white", width=0.5)
ax2.set_ylim(0,1.1); ax2.set_title("RF Model Metrics", fontweight="bold", color=NAVY)
for bar in bars:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f"{bar.get_height():.2f}", ha="center", fontsize=9)

ax3 = fig.add_subplot(2, 4, 3)
cf5 = cf_results["top_5"]
bars3 = ax3.bar(["Precision","Recall","F1"],
                [cf5["precision"],cf5["recall"],cf5["f1"]],
                color=[TEAL,NAVY,RED], edgecolor="white", width=0.5)
ax3.set_ylim(0,1.1); ax3.set_title("CF Metrics (Top-5)", fontweight="bold", color=NAVY)
for bar in bars3:
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f"{bar.get_height():.2f}", ha="center", fontsize=9)

ax4 = fig.add_subplot(2, 4, 4)
ax4.pie(sent_counts.values, labels=sent_counts.index, autopct="%1.0f%%",
        colors=[GREEN, GRAY, RED], startangle=90,
        wedgeprops={"edgecolor":"white","linewidth":1.5})
ax4.set_title("Sentiment Split", fontweight="bold", color=NAVY)

ax5 = fig.add_subplot(2, 4, 5)
ax5.plot(sizes, mr_times, marker="o", color=TEAL, lw=2, label="MapReduce")
ax5.plot(sizes, pd_times, marker="s", color=NAVY, lw=2, label="Traditional")
ax5.set_xlabel("Dataset Size"); ax5.set_ylabel("Time (ms)")
ax5.set_title("MapReduce Scalability", fontweight="bold", color=NAVY)
ax5.legend(fontsize=8)

ax6 = fig.add_subplot(2, 4, 6)
fi_s = feat_imp.sort_values()
ax6.barh(fi_s.index, fi_s.values,
         color=[TEAL if v==fi_s.max() else NAVY for v in fi_s.values], edgecolor="white")
ax6.set_title("Feature Importance", fontweight="bold", color=NAVY)

ax7 = fig.add_subplot(2, 4, 7)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=ax7, linewidths=0.5, linecolor="white")
ax7.set_title("Confusion Matrix", fontweight="bold", color=NAVY)
ax7.set_xlabel("Predicted"); ax7.set_ylabel("Actual")

ax8 = fig.add_subplot(2, 4, 8)
for lbl, col in [("At-Risk",RED),("Satisfactory",AMBER),("Excellence",GREEN)]:
    grp = df[df["label"]==lbl]
    ax8.scatter(grp["engagement_index"], grp["final_score"],
                c=col, alpha=0.4, s=10, label=lbl)
ax8.set_xlabel("Engagement Index"); ax8.set_ylabel("Final Score")
ax8.set_title("Engagement vs. Score", fontweight="bold", color=NAVY)
ax8.legend(fontsize=8, markerscale=1.5)

plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig(f"{GRAPHS_DIR}/fig6_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 6 (dashboard) saved.")

Figure 6 (dashboard) saved.


## 7. Results Summary

In [16]:
with open("results.json", "w") as f:
    json.dump(RESULTS, f, indent=2)

print("=" * 60)
print("ALL TASKS COMPLETE")
print("=" * 60)
print(f"  Records processed         : {RESULTS['n_records']}")
print(f"  At-Risk students          : {RESULTS['label_dist']['At-Risk']}")
print(f"  RF Accuracy               : {RESULTS['model_metrics']['Random Forest']['accuracy']}")
print(f"  RF F1-Score (weighted)    : {RESULTS['model_metrics']['Random Forest']['f1']}")
print(f"  GB Accuracy               : {RESULTS['model_metrics']['Gradient Boosting']['accuracy']}")
print(f"  CF Precision@5            : {RESULTS['cf_metrics']['top_5']['precision']}")
print(f"  CF Recall@5               : {RESULTS['cf_metrics']['top_5']['recall']}")
print(f"  CF F1@5                   : {RESULTS['cf_metrics']['top_5']['f1']}")
print(f"  Excellence avg compound   : {RESULTS['sentiment_by_label']['Excellence']}")
print(f"  At-Risk avg compound      : {RESULTS['sentiment_by_label']['At-Risk']}")
print(f"  Graphs saved to           : {GRAPHS_DIR}/")
print(f"  Results saved to          : results.json")

ALL TASKS COMPLETE
  Records processed         : 1200
  At-Risk students          : 171
  RF Accuracy               : 0.8667
  RF F1-Score (weighted)    : 0.859
  GB Accuracy               : 0.8767
  CF Precision@5            : 0.8
  CF Recall@5               : 0.8
  CF F1@5                   : 0.8
  Excellence avg compound   : 0.5868
  At-Risk avg compound      : -0.1602
  Graphs saved to           : graphs/
  Results saved to          : results.json
